In [ ]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import os
import time
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

In [ ]:
print("Loading Prepared Data...")
X_train = pd.read_csv('../data/processed/X_train.csv')
X_val = pd.read_csv('../data/processed/X_val.csv')
X_test = pd.read_csv('../data/processed/X_test.csv')

y_train = pd.read_csv('../data/processed/y_train.csv').values.ravel()
y_val = pd.read_csv('../data/processed/y_val.csv').values.ravel()
y_test = pd.read_csv('../data/processed/y_test.csv').values.ravel()

with open('../models/baseline_metrics.json', 'r') as f:
    baseline_metrics = json.load(f)

results = [baseline_metrics]

In [ ]:
def evaluate_model(model, name, X_train, y_train, X_val, y_val, is_nn=False):
    start_time = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start_time
    
    y_val_pred = model.predict(X_val)
    if is_nn:
        y_val_prob = model.predict_proba(X_val)[:, 1]
    else:
        y_val_prob = model.predict_proba(X_val)[:, 1]
        
    acc = accuracy_score(y_val, y_val_pred)
    prec = precision_score(y_val, y_val_pred)
    rec = recall_score(y_val, y_val_pred)
    f1 = f1_score(y_val, y_val_pred)
    auc = roc_auc_score(y_val, y_val_prob)
    
    metrics = {
        'model_name': name,
        'accuracy': float(acc),
        'precision': float(prec),
        'recall': float(rec),
        'f1_score': float(f1),
        'roc_auc': float(auc),
        'training_time': float(train_time)
    }
    
    os.makedirs('../models', exist_ok=True)
    joblib.dump(model, f'../models/{name.lower().replace(" ", "_")}.pkl')
    
    print(f"--- {name} ---")
    print(f"ROC-AUC: {auc:.4f} | F1: {f1:.4f} | Recall: {rec:.4f} | Precision: {prec:.4f}")
    
    return metrics


### Decision Tree Classifier

**Q1: What is max_depth and how does it affect the model?**
`max_depth` specifies the maximum number of decision splits the tree can make from root to leaf. A higher depth allows the model to learn more complex relationships but increases the risk of overfitting.

**Q2: What happens if max_depth is too large?**
The tree will memorize the training data (overfit), resulting in extremely high training accuracy but poor generalization to unseen validation/test data.

**Q3: How do you prevent overfitting in decision trees?**
By pruning the tree using hyperparameters like `max_depth`, `min_samples_split`, `min_samples_leaf`, and `ccp_alpha` (cost-complexity pruning).

In [ ]:
dt = DecisionTreeClassifier(max_depth=5, min_samples_split=20, random_state=42)
dt_metrics = evaluate_model(dt, 'Decision Tree', X_train, y_train, X_val, y_val)
results.append(dt_metrics)

### Random Forest Classifier

**Q1: What is an ensemble method?**
It's a technique that combines multiple machine learning models to create a more robust and accurate overarching predictive model.

**Q2: How many trees should you use? (n_estimators)**
Usually between 100 to 500. More trees reduce variance but increase computational cost. Performance plateaus after a certain number of trees.

**Q3: Extract and plot feature importance**
Plotted below in the designated section.

In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_depth=8, min_samples_split=10, random_state=42, n_jobs=-1)
rf_metrics = evaluate_model(rf, 'Random Forest', X_train, y_train, X_val, y_val)
results.append(rf_metrics)

plt.figure(figsize=(10, 8))
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]
sns.barplot(x=importances[indices][:15], y=np.array(X_train.columns)[indices][:15])
plt.title('Random Forest Feature Importance')
os.makedirs('../visualizations/model', exist_ok=True)
plt.savefig('../visualizations/model/rf_feature_importance.png')
plt.close()
print("Random Forest Feature importance parsed and plotted.")

### Gradient Boosting (XGBoost)

**Q1: What is the difference between boosting and bagging?**
Bagging (like Random Forest) trains models in parallel independently and averages them to reduce variance. Boosting (like XGBoost) trains models sequentially, where each new tree tries to correct the errors of the previous ones, reducing both bias and variance.

**Q2: What is learning_rate and why is it important?**
It scales the contribution of each tree. A lower learning rate makes the model more robust to overfitting but requires a higher `n_estimators` to achieve minimum error.

**Q3: How do you tune n_estimators to avoid overfitting?**
Using early stopping during cross-validation, which stops adding trees once the validation score hasn't improved for a set number of rounds.

In [ ]:
# Scale pos weight for imbalanced target
scale_pos_weight = (len(y_train) - sum(y_train)) / sum(y_train)
xgb = XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=5, scale_pos_weight=scale_pos_weight, random_state=42, n_jobs=-1)
xgb_metrics = evaluate_model(xgb, 'XGBoost', X_train, y_train, X_val, y_val)
results.append(xgb_metrics)

### Neural Network

**Q1: What is a hidden layer?**
A layer of nodes situated between the input and output layers where non-linear transformations and feature extraction take place.

**Q2: What happens if you use too many layers?**
The network becomes overly complex, leading to massive overfitting on tabular data, slow training times, and vanishing gradient issues unless specialized techniques are used.

**Q3: How do you prevent overfitting in neural networks?**
Using dropout, L2 regularization (weight decay), early stopping, and simplifying the network architecture (fewer neurons/layers).

In [ ]:
nn = MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu', alpha=0.01, max_iter=500, random_state=42, early_stopping=True)
nn_metrics = evaluate_model(nn, 'Neural Network', X_train, y_train, X_val, y_val, is_nn=True)
results.append(nn_metrics)

In [ ]:
# Model Comparison Section
comparison_df = pd.DataFrame(results)
if 'training_time' not in comparison_df.columns:
    comparison_df['training_time'] = [0.0] * len(comparison_df)

# Ensure columns are perfectly aligned with rubric requirements
comparison_df = comparison_df[['model_name', 'accuracy', 'precision', 'recall', 'f1_score', 'roc_auc', 'training_time']]
comparison_df.columns = ['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC', 'Training_Time']

print("\nModel Comparison:")
print(comparison_df.to_string(index=False))

comparison_df.to_csv('../models/model_comparison.csv', index=False)
print("Saved model_comparison.csv")

# Visualization
viz_df = comparison_df.melt(id_vars='Model', value_vars=['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'], 
                            var_name='Metric', value_name='Score')

plt.figure(figsize=(14, 8))
sns.barplot(x='Metric', y='Score', hue='Model', data=viz_df)
plt.title('Model Metrics Comparison')
plt.ylim(0, 1.05)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('../visualizations/model_comparison.png')
plt.close()
print("Model comparison plot saved as visualizations/model_comparison.png")